# Assignment 03: Systematic Regularization Experiments

**Module:** 04 — Neural Networks Deep Dive  
**Estimated Time:** 10-14 hours

---

## Learning Objectives

- Conduct rigorous controlled experiments measuring regularization effects
- Compare L2 (weight decay), dropout, BatchNorm, data augmentation, and early stopping
- Combine techniques and analyze interaction effects
- Create publication-quality visualizations of experimental results

### Experimental Design

- **Dataset:** CIFAR-10 with only 2,000 training samples (deliberate overfitting)
- **Base model:** 5-layer MLP with ~3.5M parameters
- **Control variable:** same optimizer (Adam/AdamW), same epochs (200), same data splits

### Key Concepts

**L2 Regularization (weight decay):** $L_{total} = L_{CE} + \lambda \sum_i w_i^2$

**Dropout:** During training, randomly zero out activations with probability $p$. Scale by $\frac{1}{1-p}$ to maintain expected values.

**Batch Normalization:** $\hat{z}_j = \frac{z_j - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}}$, then $\tilde{z}_j = \gamma_j \hat{z}_j + \beta_j$

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from collections import defaultdict

sys.path.insert(0, '../../')
from shared_utils.common import set_seed, get_device

set_seed(42)
device = get_device()
print(f'Device: {device}')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Setup complete.')

## Experimental Setup

In [ ]:
# --- Dataset: CIFAR-10, small subset to guarantee overfitting ---
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

full_train = torchvision.datasets.CIFAR10(root='../../data', train=True,
                                           download=True, transform=transform_base)
test_dataset = torchvision.datasets.CIFAR10(root='../../data', train=False,
                                             download=True, transform=transform_base)

# Take only 2000 training samples (200 per class)
np.random.seed(42)
indices_per_class = defaultdict(list)
for idx, (_, label) in enumerate(full_train):
    indices_per_class[label].append(idx)

selected = []
for cls in range(10):
    selected.extend(np.random.choice(indices_per_class[cls], 200, replace=False))

# Split: 1500 train, 500 validation
np.random.shuffle(selected)
train_indices = selected[:1500]
val_indices = selected[1500:]

train_dataset = Subset(full_train, train_indices)
val_dataset = Subset(full_train, val_indices)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

In [ ]:
# --- Base Model (deliberately large) ---
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


# Count parameters
model_test = BaseModel()
n_params = sum(p.numel() for p in model_test.parameters())
print(f'Base model parameters: {n_params:,} (~3.5M for 1,500 samples -> will overfit)')

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, test_loader,
                       optimizer, n_epochs=200, device='cpu',
                       scheduler=None):
    """Train model and track all metrics.

    Returns:
        dict with train_losses, val_losses, train_accs, val_accs,
        test_acc, best_val_acc, best_val_epoch
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }
    best_val_acc = 0.0
    best_val_epoch = 0

    for epoch in range(n_epochs):
        # Training
        model.train()
        total_loss, correct, total = 0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += inputs.size(0)

        history['train_loss'].append(total_loss / total)
        history['train_acc'].append(correct / total)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == targets).sum().item()
                val_total += inputs.size(0)

        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if history['val_acc'][-1] > best_val_acc:
            best_val_acc = history['val_acc'][-1]
            best_val_epoch = epoch

        if scheduler:
            scheduler.step()

    # Test
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            test_correct += (outputs.argmax(1) == targets).sum().item()
            test_total += inputs.size(0)

    history['test_acc'] = test_correct / test_total
    history['best_val_acc'] = best_val_acc
    history['best_val_epoch'] = best_val_epoch
    history['overfit_gap'] = history['train_acc'][-1] - history['test_acc']

    return history

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

---
## Experiment 0: Baseline (No Regularization)

Expected: training ~100%, test ~35-45%.

In [ ]:
# --- Experiment 0: Baseline ---
set_seed(42)
model_baseline = BaseModel()
optimizer_baseline = optim.Adam(model_baseline.parameters(), lr=1e-3)

print('Training baseline (no regularization)...')
results_baseline = train_and_evaluate(
    model_baseline, train_loader, val_loader, test_loader,
    optimizer_baseline, n_epochs=200, device=device
)

print(f'Final train acc: {results_baseline["train_acc"][-1]:.4f}')
print(f'Final test acc:  {results_baseline["test_acc"]:.4f}')
print(f'Overfit gap:     {results_baseline["overfit_gap"]:.4f}')
print(f'Best val acc:    {results_baseline["best_val_acc"]:.4f} (epoch {results_baseline["best_val_epoch"]})')

---
## Experiment 1: L2 Regularization (Weight Decay)

Use **AdamW** (not Adam with weight_decay). Test $\lambda \in \{10^{-5}, 10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}\}$.

In [ ]:
# --- Experiment 1: L2 / Weight Decay ---
weight_decays = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
results_l2 = {}

for wd in weight_decays:
    set_seed(42)
    model = BaseModel()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    print(f'Training with weight_decay={wd}...')
    results_l2[wd] = train_and_evaluate(
        model, train_loader, val_loader, test_loader,
        optimizer, n_epochs=200, device=device
    )
    print(f'  Test acc: {results_l2[wd]["test_acc"]:.4f}, '
          f'Overfit gap: {results_l2[wd]["overfit_gap"]:.4f}')

In [ ]:
# --- Plot: L2 sweep ---
# YOUR CODE HERE
# Plot test accuracy vs lambda (log scale x-axis)
# Mark optimal lambda
pass

**At what $\lambda$ does test accuracy peak? What happens to training accuracy as $\lambda$ increases?**

*YOUR ANSWER HERE*

**Why AdamW instead of Adam with weight_decay?**

*YOUR ANSWER HERE*

---
## Experiment 2: Dropout

Test dropout rates: $\{0.1, 0.2, 0.3, 0.5, 0.7\}$.

In [ ]:
class DropoutModel(nn.Module):
    def __init__(self, dropout_rate: float = 0.5):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 1024),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# --- Experiment 2: Dropout ---
dropout_rates = [0.1, 0.2, 0.3, 0.5, 0.7]
results_dropout = {}

for dr in dropout_rates:
    set_seed(42)
    model = DropoutModel(dropout_rate=dr)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    print(f'Training with dropout={dr}...')
    results_dropout[dr] = train_and_evaluate(
        model, train_loader, val_loader, test_loader,
        optimizer, n_epochs=200, device=device
    )
    print(f'  Test acc: {results_dropout[dr]["test_acc"]:.4f}')

In [ ]:
# --- Plot: Dropout sweep ---
# YOUR CODE HERE
pass

---
## Experiment 3: Batch Normalization

In [ ]:
class BNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# --- Experiment 3: BatchNorm ---
set_seed(42)
model_bn = BNModel()
optimizer_bn = optim.Adam(model_bn.parameters(), lr=1e-3)

print('Training with BatchNorm...')
results_bn = train_and_evaluate(
    model_bn, train_loader, val_loader, test_loader,
    optimizer_bn, n_epochs=200, device=device
)
print(f'Test acc: {results_bn["test_acc"]:.4f}, Overfit gap: {results_bn["overfit_gap"]:.4f}')

**Does BatchNorm primarily regularize or help optimization?**

*YOUR ANSWER HERE*

---
## Experiment 4: Data Augmentation

In [ ]:
# --- Experiment 4: Data Augmentation ---
augmentation_configs = {
    '4a: Flip': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ]),
    '4b: Flip+Crop': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ]),
    '4c: Flip+Crop+Color': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ]),
    '4d: +Rotation': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ]),
}

results_aug = {}
for name, aug_transform in augmentation_configs.items():
    set_seed(42)
    # Create augmented dataset
    aug_full = torchvision.datasets.CIFAR10(root='../../data', train=True,
                                             download=False, transform=aug_transform)
    aug_train = Subset(aug_full, train_indices)
    aug_loader = DataLoader(aug_train, batch_size=64, shuffle=True)

    model = BaseModel()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    print(f'Training with {name}...')
    results_aug[name] = train_and_evaluate(
        model, aug_loader, val_loader, test_loader,
        optimizer, n_epochs=200, device=device
    )
    print(f'  Test acc: {results_aug[name]["test_acc"]:.4f}')

**Which augmentation has the biggest effect? Diminishing returns?**

*YOUR ANSWER HERE*

---
## Experiment 5: Early Stopping

In [ ]:
# --- Experiment 5: Early Stopping ---
# Use baseline results and find best stopping epoch for each patience
patience_values = [5, 10, 20, 50]

for patience in patience_values:
    val_losses = results_baseline['val_loss']
    best_loss = float('inf')
    counter = 0
    stop_epoch = len(val_losses) - 1

    for epoch, vl in enumerate(val_losses):
        if vl < best_loss:
            best_loss = vl
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            stop_epoch = epoch
            break

    val_acc_at_stop = results_baseline['val_acc'][stop_epoch]
    print(f'Patience={patience}: stop at epoch {stop_epoch}, val acc = {val_acc_at_stop:.4f}')

---
## Experiment 6: Combinations

In [ ]:
# --- Experiment 6: Combinations ---
# YOUR CODE HERE
# 6a: Best L2 + best dropout
# 6b: Best L2 + BatchNorm
# 6c: Best dropout + data augmentation (4c)
# 6d: BatchNorm + augmentation + L2
# 6e: All: BN + aug + L2 + early stopping
# 6f: Your best combination
#
# For each, train and record results
pass

---
## Required Visualizations

In [ ]:
# --- Plot 1: Overfitting Gap Bar Chart ---
# YOUR CODE HERE
# Bar chart: (train acc - test acc) for baseline, best L2, best dropout,
# BN, best augmentation, best combination
pass

In [ ]:
# --- Plot 2: Learning Curves (2x2 grid) ---
# YOUR CODE HERE
# Top-left: Training loss (baseline, L2, dropout, BN)
# Top-right: Validation loss
# Bottom-left: Training accuracy
# Bottom-right: Validation accuracy
pass

In [ ]:
# --- Plot 3: L2 sweep ---
# YOUR CODE HERE
pass

In [ ]:
# --- Plot 4: Dropout sweep ---
# YOUR CODE HERE
pass

In [ ]:
# --- Plot 5: Augmentation progression ---
# YOUR CODE HERE
pass

In [ ]:
# --- Plot 6: Final comparison (all experiments) ---
# YOUR CODE HERE
# Comprehensive bar chart of test accuracy, sorted
pass

---
## Written Analysis (800-1200 words)

### 1. Baseline Characterization

*YOUR ANALYSIS HERE*

### 2. Individual Technique Comparison

*YOUR ANALYSIS HERE*

### 3. Interaction Effects

*YOUR ANALYSIS HERE*

### 4. The Role of Data Augmentation

*YOUR ANALYSIS HERE*

### 5. BatchNorm as Regularizer

*YOUR ANALYSIS HERE*

### 6. Practical Recommendations

*YOUR ANALYSIS HERE*

### 7. Limitations

*YOUR ANALYSIS HERE*

---
## Summary Table

In [ ]:
# --- Summary table ---
# YOUR CODE HERE
# Create a table with columns:
# Experiment | Train Acc | Val Acc | Test Acc | Overfit Gap | Best Val Epoch
pass